# **Notebook 3: Baseline Model Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] Internet access (to download the model)
- [ ] GPU runtime enabled (Runtime → Change runtime type → T4 GPU)

**Files this notebook will CREATE:**
- [ ] `outputs.json` — `test_query`, `ground_truth`, `baseline_output` _(Required by NB4, NB5, NB7)_

---

## **Stage 3: Solution V1 (Retrieval-Assisted Generation)**

### **Task 3.1: Establish Baseline Performance**

#### **3.1.1 Execute Baseline Inference [2 marks]**
**The Task:** Load the pre-trained base model in 4-bit quantization and generate a response to an ambiguous shipping-delay query without any context.

**Hints & Tips:**
* Use `do_sample=False` for deterministic output. Do NOT pair `temperature=0.0` with `do_sample=False` — it throws a deprecation warning. Use `temperature=None, top_p=None`.
* `BitsAndBytesConfig(load_in_4bit=True)` shrinks the 1.5B model to ~750MB VRAM.
* `max_new_tokens=120` gives room for a complete answer.

**Model Selection:**
* **Qwen/Qwen2.5-1.5B-Instruct** (recommended) — must match what you used in NB2.
* **TinyLlama-1.1B-Chat** — lighter, weaker structured output.
* **Llama-3-8B-Instruct** — best quality, may OOM on free T4 during fine-tuning.

**Learner Inference:** This establishes your zero-shot baseline. Every later improvement is measured against this exact output.

In [2]:
# YOUR CODE HERE
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# The instructions suggests 4-bit (bitsandbytes) on a CUDA T4.
# I am running this project on Apple Silicon (M1) → So no CUDA, no bitsandbytes.
# I am taking Mac-friendly path: load in float16 on MPS instead.
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
device = "mps" if torch.backends.mps.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
).to(device)

model.eval()
print(f"Model loaded on: {device}")

/Users/abinas/Documents/projects/machine-learning/capstone-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 175.58it/s]


Model loaded on: mps


In [3]:
# YOUR CODE HERE
test_query = "My order is late, where is it??"

messages = [{"role": "user", "content": test_query}]
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        temperature=None,
        top_p=None,
    )

baseline_output = tokenizer.decode(
    output_ids[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print(baseline_output)

I'm sorry to hear that your order is late. Please check the following steps:

1. Check if you have received any notifications or reminders about the delivery status.
2. Contact the seller or courier company directly to inquire about the delay and get an update on when your order will be delivered.
3. If possible, try to track the package yourself using the tracking number provided in your order confirmation.

If you still cannot find out the reason for the delay, please provide more information so I can assist you better.


#### **3.1.2 Evaluate Baseline Quality [2 marks]**
**The Task:** Assess the baseline output for factual inaccuracies against the ground-truth SOP rule.

**Hints & Tips:**
* Compare against the known rule: "Domestic orders deliver within 3-7 business days."
* Did the model invent a timeline? Mention a non-existent tracking system or department?
* Document every hallucination — it justifies Stages 3 and 4.

**Learner Inference:** This hallucination is exactly why you build Stage 3 (a database) and Stage 4 (a router).

In [4]:
# YOUR CODE HERE
ground_truth = "Domestic orders deliver within 3-7 business days."

print("=== BASELINE HALLUCINATION ASSESSMENT ===\n")
print("SOP RULE:", ground_truth, "\n")

print("Issues found in baseline output:")
print("1. Ignored our 3-7 business day timeline entirely.")
print("2. Told the customer to contact the 'seller/courier' "
      "— we are the company here; wrong redirection.")
print("3. Offloaded tracking onto the customer instead of "
      "offering a carrier trace (our real procedure).")
print("4. Generic advice, zero grounding in our actual policy.")
print("\nCONCLUSION: Fluent but policy-blind.")

=== BASELINE HALLUCINATION ASSESSMENT ===

SOP RULE: Domestic orders deliver within 3-7 business days. 

Issues found in baseline output:
1. Ignored our 3-7 business day timeline entirely.
2. Told the customer to contact the 'seller/courier' — we are the company here; wrong redirection.
3. Offloaded tracking onto the customer instead of offering a carrier trace (our real procedure).
4. Generic advice, zero grounding in our actual policy.

CONCLUSION: Fluent but policy-blind.


---
## Save Artifacts for Downstream Notebooks

**IMPORTANT:** Saves the baseline output. Notebooks 4, 5, and 7 depend on this file.

In [5]:
# YOUR CODE HERE
import json

outputs = {
    "test_query": test_query,
    "ground_truth": ground_truth,
    "baseline_output": baseline_output,
}

with open("outputs.json", "w") as f:
    json.dump(outputs, f, indent=2)

print("Saved outputs.json")

with open("outputs.json") as f:
    check = json.load(f)
print("Keys saved:", list(check.keys()))


Saved outputs.json
Keys saved: ['test_query', 'ground_truth', 'baseline_output']


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 4.**

- [ ] Base model loaded in 4-bit without errors
- [ ] Baseline output generated for `test_query`
- [ ] Hallucination assessment documented
- [ ] **`outputs.json` saved** with `test_query`, `ground_truth`, `baseline_output` ← _CRITICAL for NB4, 5, 7_
- [ ] GPU runtime enabled

**If any item is unchecked, fix it before moving on.**